# Query Masking Ablations

This notebook recreates the query maskings for different random seeds and computes the overlap between them.

In [1]:
import numpy as np
from pathlib import Path

# Set parameters
frac = 0.05  # Fraction of maskable tokens to replace (adjust as needed)
seeds = [0, 1]  # Hardcoded list of seeds
fasta_file = Path("../data/ZnT8/ZnT8.fasta")  # Path to the unmasked sequence

In [2]:
# Read the original unmasked sequence
lines = fasta_file.read_text().splitlines()
unmasked_seq = ""
for i in range(0, len(lines), 2):
    if lines[i].startswith(">"):
        unmasked_seq = lines[i + 1].strip()
        break

print(f"Unmasked sequence length: {len(unmasked_seq)}")

Unmasked sequence length: 369


In [3]:
# Re-create maskings
ACID_TOKENS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ-")

def get_maskable_indices(query_seq):
    return [
        k for k, char in enumerate(query_seq)
        if char in ACID_TOKENS and char not in ("-", "X")
    ]

maskable_indices = get_maskable_indices(unmasked_seq)
n_mask = int(len(maskable_indices) * frac)

masked_sequences = {}        # Maps seed -> masked sequence (string)
masked_positions = {}        # Maps seed -> set of masked indices
visualization_sequences = {} # Maps seed -> sequence with '#' for masked positions

for seed in seeds:
    rng = np.random.default_rng(seed)
    
    chosen = []
    if n_mask > 0:
        chosen = rng.choice(maskable_indices, n_mask, replace=False)
        
    # Create actual masked sequence (Alanine substitution)
    seq_list = list(unmasked_seq)
    for k in chosen:
        seq_list[k] = "A"
    masked_sequences[seed] = "".join(seq_list)
    
    # Create visualization sequence (with '#')
    vis_list = list(unmasked_seq)
    for k in chosen:
        vis_list[k] = "#"
    visualization_sequences[seed] = "".join(vis_list)
    
    masked_positions[seed] = set(chosen)

In [4]:
# Calculate overlaps
if not seeds:
    overlap_positions = set()
else:
    overlap_positions = set(masked_positions[seeds[0]])
    for s in seeds[1:]:
        overlap_positions.intersection_update(masked_positions[s])

print(f"Number of overlaps for seeds {seeds}: {len(overlap_positions)}")

Number of overlaps for seeds [0, 1]: 2


In [5]:
# Display alignment
print("1) Original query:")
print(unmasked_seq)
print("\n" + "="*80 + "\n")

print("2) Masked sequences:")
for seed in seeds:
    print(f"Seed {seed}:")
    print(visualization_sequences[seed])
    print()
print("="*80 + "\n")

print("3) Overlap:")
overlap_str = ["-"] * len(unmasked_seq)
for pos in overlap_positions:
    overlap_str[pos] = "#"
print("".join(overlap_str))

1) Original query:
MEFLERTYLVNDKAAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEVVGGHIAGSLAVVTDAAHLLIDLTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMIIVSSCAVAANIVLTVVLHQRCLGHNHKEVQANASVRAAFVHALGDLFQSISVLISALIIYFKPEYKIADPICTFIFSILVLASTITILKDFSILLMEGVPKSLNYSGVKELILAVDGVLSVHSLHIWSLTMNQVILSAHVATAASRDSQVVRREIAKALSKSFTMHSLTIQMESPVDQDPDCLFCEDPCD


2) Masked sequences:
Seed 0:
MEFLE#TYLVNDKA#KMYAFTLESVE#QQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKG#NEYAYAKWKLCSASAICFIFMIAEVVGGHIA#SLAVVTDAAHLLI#LTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMII#SS#AVAANIVLTVVLHQRC#GHNHKEVQANASVRAAFVHA#GD#FQSISVL#SA#IIYFKPEYKIADPICTFIFSILVLASTITIL#DFSILLMEGVPKSLNYSGVKELILA#DGVLS#HSLHIWSLTMNQVILSAHVATAASRDSQVVR#EIAKALSKSFTMHSLTIQMESPV#QDPDCLFCEDPCD

Seed 1:
MEFLERTYLVND#AAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMY#CHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEV#GGHI#GSLA#VTDAAHLLIDLT#FLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILC#WV#TGVLVYLACERL#YPDYQIQATVMII#SSCAVAANIVLTVVLHQRCL

In [17]:
def process_sequences(convert_to_mask, aa_sequence_reference=None, input_mask=None):
    # 1. Validate lengths of all provided inputs
    provided_inputs = [convert_to_mask]
    if aa_sequence_reference is not None:
        provided_inputs.append(aa_sequence_reference)
    if input_mask is not None:
        provided_inputs.append(input_mask)
        
    expected_length = len(provided_inputs[0])
    for seq in provided_inputs:
        if len(seq) != expected_length:
            raise ValueError("All provided inputs must have the exact same length.")

    # 2. Process the primary mask
    # Replace every character that is NOT a '#' with a '-'
    output_mask = "".join(['#' if char == '#' else '-' for char in convert_to_mask])
    
    # 3. Process the optional reference sequence and input mask
    output_reference = None
    if aa_sequence_reference is not None and input_mask is not None:
        output_reference_list = list(aa_sequence_reference)
        for i, char in enumerate(input_mask):
            if char == '#':
                output_reference_list[i] = 'A'
        output_reference = "".join(output_reference_list)
        
    # Return both if optional inputs were provided, otherwise just return the first output
    if output_reference is not None:
        return output_mask, output_reference
    
    return output_mask

# ==========================================
# Inputs
# ==========================================

convert_to_mask = "MEFLERTYLVND#AAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMY#CHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEV#GGHI#GSLA#VTDAAHLLIDLT#FLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILC#WV#TGVLVYLACERL#YPDYQIQATVMII#SSCAVAANIVLTVVLHQRCLG#NHKEVQANASVRAAFVHALGDLFQSISVLISAL#IYFKPEYKIADPICTFIFSILVLASTITIL#DFSILLMEGVPKSLNYSGVKELILAV#GVLSVHS#HIWSLTMNQVI#SAHVATAASRDSQVVRREIAKA#SK#FTMHSLTIQMESPVDQDPDCLFCEDPCD"

aa_sequence_reference = "MEFLERTYLVNDKAAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEVVGGHIAGSLAVVTDAAHLLIDLTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMIIVSSCAVAANIVLTVVLHQRCLGHNHKEVQANASVRAAFVHALGDLFQSISVLISALIIYFKPEYKIADPICTFIFSILVLASTITILKDFSILLMEGVPKSLNYSGVKELILAVDGVLSVHSLHIWSLTMNQVILSAHVATAASRDSQVVRREIAKALSKSFTMHSLTIQMESPVDQDPDCLFCEDPCD"

input_mask = "-----------------------------------------------------------------------------------------#----#----#--------------------------------------------------------------------------------#---------------------#---------------------------------#------------------------------#----------------------------------#------------------------------------------------------------------"



# ==========================================
# Generate Output 3 (Left and Right halves)
# ==========================================

# 1. Get the full mask (all '-' except for '#')
full_mask = process_sequences(convert_to_mask)

# 2. Find all indices where '#' occurs
hash_indices = [i for i, char in enumerate(full_mask) if char == '#']

# 3. Split the indices in half
half_point = len(hash_indices) // 2
left_indices = set(hash_indices[:half_point])
right_indices = set(hash_indices[half_point:])

# 4. Create the left and right masks
left_mask = "".join(['#' if i in left_indices else '-' for i in range(len(full_mask))])
right_mask = "".join(['#' if i in right_indices else '-' for i in range(len(full_mask))])

# 5. Generate output2 for both masks
_, left_out2 = process_sequences(convert_to_mask, aa_sequence_reference, left_mask)
_, right_out2 = process_sequences(convert_to_mask, aa_sequence_reference, right_mask)

# 6. Print the results in the requested format



# Calling with the optional inputs:
out1, out2 = process_sequences(convert_to_mask, aa_sequence_reference, input_mask)
print("Output 1 (With optional inputs):\n", out1)
print("\nOutput 2 (With optional inputs):\n", out2)
print("\n")




print("Output 3")
print("\nLeft part:")
print(left_mask)
print(left_out2)

print("\nRight part:")
print(right_mask)
print(right_out2)







# convert_to_mask = "MEFLERTYLVND#AAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMY#CHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEV#GGHI#GSLA#VTDAAHLLIDLT#FLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILC#WV#TGVLVYLACERL#YPDYQIQATVMII#SSCAVAANIVLTVVLHQRCLG#NHKEVQANASVRAAFVHALGDLFQSISVLISAL#IYFKPEYKIADPICTFIFSILVLASTITIL#DFSILLMEGVPKSLNYSGVKELILAV#GVLSVHS#HIWSLTMNQVI#SAHVATAASRDSQVVRREIAKA#SK#FTMHSLTIQMESPVDQDPDCLFCEDPCD"


# aa_sequence_reference = "MEFLERTYLVNDKAAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEVVGGHIAGSLAVVTDAAHLLIDLTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMIIVSSCAVAANIVLTVVLHQRCLGHNHKEVQANASVRAAFVHALGDLFQSISVLISALIIYFKPEYKIADPICTFIFSILVLASTITILKDFSILLMEGVPKSLNYSGVKELILAVDGVLSVHSLHIWSLTMNQVILSAHVATAASRDSQVVRREIAKALSKSFTMHSLTIQMESPVDQDPDCLFCEDPCD"

# # input_mask = "------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------#--------------------------------------------------------------------------------------#-----------------------------------------------------------------------------------------------------"
# input_mask = "-----------------------------------------------------------------------------------------#----#----#--------------------------------------------------------------------------------#---------------------#---------------------------------#------------------------------#----------------------------------#------------------------------------------------------------------"


# # Calling with just the primary input:
# # out1 = process_sequences(convert_to_mask)
# # print("Output 1 (Single input):\n", out1, "\n")

# # Calling with the optional inputs:
# out1, out2 = process_sequences(convert_to_mask, aa_sequence_reference, input_mask)
# print("Output 1 (With optional inputs):\n", out1)
# print("\nOutput 2 (With optional inputs):\n", out2)


Output 1 (With optional inputs):
 ------------#--------------------------------------#-------------------------------------#----#----#------------#-------------------------------------#--#------------#-------------#---------------------#---------------------------------#------------------------------#--------------------------#-------#-----------#----------------------#--#----------------------------

Output 2 (With optional inputs):
 MEFLERTYLVNDKAAKMYAFTLESVELQQKPVNKDQCPRERPEELESGGMYHCHSGSKPTEKGANEYAYAKWKLCSASAICFIFMIAEVAGGHIAGSLAAVTDAAHLLIDLTSFLLSLFSLWLSSKPPSKRLTFGWHRAEILGALLSILCIWVVTGVLVYLACERLLYPDYQIQATVMIIASSCAVAANIVLTVVLHQRCLGANHKEVQANASVRAAFVHALGDLFQSISVLISALAIYFKPEYKIADPICTFIFSILVLASTITILADFSILLMEGVPKSLNYSGVKELILAVDGVLSVHSAHIWSLTMNQVILSAHVATAASRDSQVVRREIAKALSKSFTMHSLTIQMESPVDQDPDCLFCEDPCD


Output 3

Left part:
------------#--------------------------------------#-------------------------------------#----#----#------------#-------------------------------------#--#------------#-